In [1]:
import torch 
import torch.nn as nn
from torch.nn.functional import linear

# Symmetric vs Asymmetric Quantization — Full Reference

## 1. What Quantization Is Solving

Quantization maps a wide range of real (floating-point) numbers `r` onto a small set of integers `q`, using two learned parameters:

- **scale (`s`)** — how much real-number "distance" one integer step represents
- **zero-point (`z`)** — which integer represents the real value `0.0`

The two schemes differ entirely in **how `z` is chosen**.

---

## 2. Symmetric Quantization

### Core idea
The real-number range is forced to be **centered on zero**: `[-max, +max]`. Because zero in float always lands exactly on zero in int, **no zero-point is needed** — `z = 0` by definition.

### Formulas

**Scale:**
$$
s = \frac{\max(|r|)}{q_{max}}
$$

**Quantize:**
$$
q = \text{round}\left(\frac{r}{s}\right), \quad \text{clamped to } [-q_{max},\ q_{max}]
$$

**Dequantize:**
$$
r \approx s \cdot q
$$

For **INT8**: `q_max = 127`, so the usable integer range is `[-127, 127]` (one bucket, `-128`, is typically left unused to keep the range symmetric).

### Key property
`max(|r|)` (absolute value) is used — not the raw max — because the *farthest point from zero in either direction* determines how much range needs to be covered on both sides equally.

---

## 3. Asymmetric Quantization

### Core idea
The real-number range is **not** forced through zero. It maps the **true range** `[min(r), max(r)]`, wherever it actually sits, onto the full unsigned integer range `[0, q_max]`. Since zero may not fall on an integer boundary anymore, a **zero-point offset (`z`)** is needed to track where `0.0` lands in integer space.

### Formulas

**Scale:**
$$
s = \frac{\max(r) - \min(r)}{q_{max} - q_{min}}
$$

**Zero-point:**
$$
z = \text{round}\left(q_{min} - \frac{\min(r)}{s}\right), \quad \text{clamped to } [q_{min},\ q_{max}]
$$

**Quantize:**
$$
q = \text{round}\left(\frac{r}{s}\right) + z, \quad \text{clamped to } [q_{min},\ q_{max}]
$$

**Dequantize:**
$$
r \approx s \cdot (q - z)
$$

For **UINT8**: `q_min = 0`, `q_max = 255`.

---

## 4. Side-by-Side Comparison

| Aspect | Symmetric | Asymmetric |
|---|---|---|
| Zero-point (`z`) | Always `0` | Non-zero, computed per tensor/channel |
| Integer range used | `[-127, 127]` (INT8) — signed | `[0, 255]` (UINT8) — unsigned |
| What defines the scale | `max(\|r\|)` — distance from zero | `max(r) - min(r)` — full real span |
| Storage overhead | Scale only | Scale **and** zero-point |
| Compute cost | Lower — pure multiply | Slightly higher — extra add/subtract per op |
| Best suited for | Data naturally centered around 0 (e.g. weights) | Data skewed / one-sided (e.g. post-ReLU activations) |
| Wasted range risk | High, if data is skewed (half the buckets go unused) | Low — full integer range maps to the real data span |
| Hardware fusion | Simpler to fuse into fast INT8 matmul kernels | More complex, but well supported in TensorRT / most inference engines |

---

## 5. When to Use Which

| Situation | Recommended scheme | Why |
|---|---|---|
| Neural network **weights** | Symmetric | Weights are typically roughly Gaussian, centered near 0 — no benefit from an offset, and simpler math |
| **Activations after ReLU / GELU** | Asymmetric | These are one-sided (all ≥ 0, or heavily skewed) — symmetric would waste half the integer range |
| Data with a **known hard bound at zero** (e.g. pixel intensities `[0, 255]`) | Asymmetric | The natural range already starts at 0, so asymmetric fits without wasted buckets |
| You need **max inference speed**, minor accuracy loss acceptable | Symmetric | Fewer operations, easier for hardware to optimize (no offset term) |
| You need **max accuracy on skewed data**, and can tolerate a bit more compute | Asymmetric | Uses the full integer resolution on the actual data range |
| Common real-world hybrid (TensorRT, PyTorch quantization, AWQ) | **Symmetric weights + Asymmetric activations** | Combines simplicity where it doesn't cost accuracy, with precision where it matters most |

---

## 6. Worked Calculation Examples

### Example A — Symmetric, tensor `r = [-9.0, 1.0, 2.0, 3.0]`

**Step 1 — find scale:**
$$
s = \frac{\max(|-9|,|1|,|2|,|3|)}{127} = \frac{9.0}{127} \approx 0.0709
$$

**Step 2 — quantize each value** (`q = round(r/s)`):

| `r` | `r / s` | `q` (rounded) |
|---|---|---|
| -9.0 | -126.98 | **-127** |
| 1.0 | 14.10 | **14** |
| 2.0 | 28.21 | **28** |
| 3.0 | 42.31 | **42** |

**Step 3 — dequantize** (`r ≈ s·q`):

| `q` | `s·q` | Recovered `r` | Original `r` | Error |
|---|---|---|---|---|
| -127 | 0.0709 × -127 | -9.00 | -9.0 | ~0.00 |
| 14 | 0.0709 × 14 | 0.99 | 1.0 | 0.01 |
| 28 | 0.0709 × 28 | 1.98 | 2.0 | 0.02 |
| 42 | 0.0709 × 42 | 2.98 | 3.0 | 0.02 |

Note the extreme value (`-9.0`) is recovered almost perfectly since it defined the scale; the smaller values pick up slightly more rounding error.

---

### Example B — Asymmetric, same tensor `r = [-9.0, 1.0, 2.0, 3.0]`

**Step 1 — find scale (full range, not just abs max):**
$$
s = \frac{\max(r) - \min(r)}{255 - 0} = \frac{3.0 - (-9.0)}{255} = \frac{12.0}{255} \approx 0.0471
$$

**Step 2 — find zero-point:**
$$
z = \text{round}\left(0 - \frac{-9.0}{0.0471}\right) = \text{round}(191.1) = 191
$$

**Step 3 — quantize each value** (`q = round(r/s) + z`):

| `r` | `r/s` | `+ z` | `q` |
|---|---|---|---|
| -9.0 | -191.08 | +191 | **0** |
| 1.0 | 21.23 | +191 | **212** |
| 2.0 | 42.46 | +191 | **233** |
| 3.0 | 63.69 | +191 | **255** |

**Step 4 — dequantize** (`r ≈ s·(q - z)`):

| `q` | `q - z` | `s·(q-z)` | Recovered `r` | Original `r` |
|---|---|---|---|---|
| 0 | -191 | 0.0471 × -191 | -9.00 | -9.0 |
| 212 | 21 | 0.0471 × 21 | 0.99 | 1.0 |
| 233 | 42 | 0.0471 × 42 | 1.98 | 2.0 |
| 255 | 64 | 0.0471 × 64 | 3.01 | 3.0 |

Here the **entire** integer range `[0, 255]` is used to cover the real span `[-9, 3]`, giving a smaller scale (`0.0471` vs `0.0709`) — meaning finer resolution and generally lower error across the board, especially since none of the range is "wasted" on values that never occur.

---

## 7. Intuition Summary

- **Symmetric** = "How far does the data reach from zero, in the worst direction?" → mirror that distance on both sides.
- **Asymmetric** = "What is the *actual* min-to-max span of the data?" → stretch the full integer range across exactly that span, wherever it sits.

If your data is already centered on zero, both methods converge to roughly the same accuracy — symmetric just does it with less overhead. If your data is skewed (like post-activation values), asymmetric avoids wasting integer buckets on values that can never occur, giving a tighter, more accurate mapping.

In [33]:
## Implementing Quantization in Pytorch
from typing import Tuple, List, Union
from torch import Tensor

def symmetric_quantization(r: torch.Tensor, n_bits: int, dim: int = None) -> Tuple[Tensor, Tensor]:
    """
    Symmetric affine quantization: zero_point is always 0.
    Range is [-2^(n_bits-1)+1, 2^(n_bits-1)-1]  (we use the "restricted" range so
    that -qmax*scale and +qmax*scale are both representable — this is
    what PyTorch/TensorRT do for symmetric weight quant).

    Args:
        r: tensor to quantize
        num_bits: bit width (e.g. 8 for int8)
        dim: if given, compute one scale per slice along this dim
             (per-channel quantization). If None -> per-tensor.
        q = round(r / s)
        r = s * q
    Returns:
        q: quantized integer tensor (stored as int8/int32 for generality)
        scale: the scale(s) used
    """
    q_max = (2 ** (n_bits - 1)) - 1
    if not dim:
        r_max = r.abs().max().clamp(min=1e-8)
        scale = r_max / q_max
        scale = torch.clamp(scale, min=1e-8)
    else:
        # per-channel: reduce over all dims except `dim`
        reduce_dims = [d for d in range(r.dim()) if d != dim]
        max_val = r.abs().amax(dim=reduce_dims, keepdim=True).clamp(min=1e-8)
        scale = r_max / q_max
    q = torch.round(r / scale).clamp(-q_max - 1, q_max)
    return q.to(torch.int8), scale 

def symmetric_dequantize(q: torch.Tensor, scale: torch.Tensor):
    return q.float() * scale

def quantization_error(r: torch.Tensor, q: torch.Tensor, scale: torch.Tensor) -> Tensor:
    """
    Computes the quantization error (Mean Squared Error) between the
    original tensor and its dequantized reconstruction.

    Args:
        r: original real-valued tensor
        q: quantized integer tensor (output of symmetric_quantization)
        scale: scale(s) used during quantization (output of symmetric_quantization)

    Returns:
        Scalar tensor representing the MSE between r and its dequantized version.
    """
    r_reconstructed = symmetric_dequantize(q, scale)
    return torch.mean((r - r_reconstructed) ** 2)

def aquantization_error(r: torch.Tensor, q: torch.Tensor, scale: torch.Tensor, zero_point) -> Tensor:
    """
    Computes the quantization error (Mean Squared Error) between the
    original tensor and its dequantized reconstruction.

    Args:
        r: original real-valued tensor
        q: quantized integer tensor (output of symmetric_quantization)
        scale: scale(s) used during quantization (output of symmetric_quantization)

    Returns:
        Scalar tensor representing the MSE between r and its dequantized version.
    """
    r_reconstructed = asymmetric_dequantize(q, scale, zero_point)
    return torch.mean((r - r_reconstructed) ** 2)

def quantization_error_mae(r, q, scale):
    r_reconstructed = symmetric_dequantize(q, scale)
    return torch.mean(torch.abs(r - r_reconstructed))


def asymmetric_quantization(r: torch.Tensor, n_bits: int, dim: int = None) -> Tuple[Tensor, Tensor, Tensor]:
    """
    Full affine quantization with a zero_point. Used for activations
    whose distribution isn't centered at 0 (e.g. post-ReLU >= 0).

    q = round(x/scale) + zero_point,   scale = (max-min)/(qmax-qmin)
    """
    q_min, q_max = 0, 2 ** (n_bits) - 1
    r_max, r_min = r.max(), r.min()
    scale = (r_max - r_min) / (q_max - q_min)
    zero_point = q_min - torch.round(min_val / scale)
    zero_point = torch.clamp(zero_point, q_min, q_max)

    q = torch.round(r / scale) + zero_point
    q = torch.clamp(q, q_min, q_max)
    return q.to(torch.uint8), scale, zero_point

def asymmetric_dequantize(q: torch.Tensor, scale: torch.Tensor, zero_point: torch.Tensor):
    return (q.to(torch.float32) - zero_point) * scale

In [34]:
torch.manual_seed(0)  # optional, for reproducibility

params = torch.empty(20).uniform_(-50, 150)
params[0] = params.max() + 1
params[1] = params.min() - 1
params[2] = 0

params = torch.round(params * 100) / 100  # round to 2 decimal places
print(params)

tensor([130.2900, -46.5300,   0.0000, -23.5900,  11.4800,  76.8200,  48.0200,
        129.2900,  41.1300,  76.4600,  19.7800,  30.3400, -45.5300, -16.2300,
          8.7800,  53.7000,  89.5300, 110.0000, -17.7900,   6.4500])


In [35]:
quantized, scale = symmetric_quantization(r=params, n_bits=8)
print(quantized)
print(scale)

tensor([127, -45,   0, -23,  11,  75,  47, 126,  40,  75,  19,  30, -44, -16,
          9,  52,  87, 107, -17,   6], dtype=torch.int8)
tensor(1.0259)


In [36]:
dequantized = symmetric_dequantize(*symmetric_quantization(r=params, n_bits=8))

In [37]:
torch.mean((params - dequantized)**2)

tensor(0.0797)

In [38]:
quantization_error(params, quantized, scale)

tensor(0.0797)

In [39]:
quantized, scale, zero_point = asymmetric_quantization(r=params, n_bits=8)
print(quantized)
print(scale)
print(zero_point)

tensor([255,   0,  67,  33,  84, 178, 136, 253, 126, 177,  96, 111,   1,  44,
         80, 144, 196, 226,  41,  76], dtype=torch.uint8)
tensor(0.6934)
tensor(67.)


In [40]:
params

tensor([130.2900, -46.5300,   0.0000, -23.5900,  11.4800,  76.8200,  48.0200,
        129.2900,  41.1300,  76.4600,  19.7800,  30.3400, -45.5300, -16.2300,
          8.7800,  53.7000,  89.5300, 110.0000, -17.7900,   6.4500])

In [41]:
dequantized = asymmetric_dequantize(*asymmetric_quantization(r=params, n_bits=8))
dequantized

tensor([130.3614, -46.4586,   0.0000, -23.5760,  11.7880,  76.9687,  47.8454,
        128.9746,  40.9113,  76.2753,  20.1089,  30.5101, -45.7652, -15.9485,
          9.0144,  53.3927,  89.4501, 110.2525, -18.0287,   6.2407])

In [42]:
torch.mean((params - dequantized)**2)

tensor(0.0466)

In [44]:
aquantization_error(params, quantized, scale, zero_point)

tensor(0.0466)

In [45]:
"""
Testing quantization using PyTorch's BUILT-IN quantization API
(not from-scratch) -- torch.quantize_per_tensor / per_channel,
and real dynamic quantization of an nn.Linear layer.
"""
import torch
import torch.nn as nn

r = torch.tensor([130.29, -46.53, 0.0, -23.59, 11.48, 76.82, 48.02,
                   129.29, 41.13, 76.46, 19.78, 30.34, -45.53, -16.23,
                   8.78, 53.70, 89.53, 110.00, -17.79, 6.45])

# ---------------------------------------------------------------
# 1) torch.quantize_per_tensor -- real quantized tensor (qint8)
# ---------------------------------------------------------------
scale = r.abs().max().item() / 127
zero_point = 0
q_tensor = torch.quantize_per_tensor(r, scale=scale, zero_point=zero_point,
                                      dtype=torch.qint8)

print("=== torch.quantize_per_tensor (symmetric-style) ===")
print("scale used      :", scale)
print("q_tensor dtype   :", q_tensor.dtype)
print("q_tensor.int_repr():", q_tensor.int_repr())
print("dequantized      :", q_tensor.dequantize())
print()

# ---------------------------------------------------------------
# 2) torch.quantize_per_tensor -- ASYMMETRIC (uint8, non-zero zp)
# ---------------------------------------------------------------
min_val, max_val = r.min().item(), r.max().item()
scale_a = (max_val - min_val) / 255
zero_point_a = int(round(-min_val / scale_a))

q_tensor_asym = torch.quantize_per_tensor(r, scale=scale_a,
                                           zero_point=zero_point_a,
                                           dtype=torch.quint8)

print("=== torch.quantize_per_tensor (asymmetric, quint8) ===")
print("scale, zero_point:", scale_a, zero_point_a)
print("int_repr()        :", q_tensor_asym.int_repr())
print("dequantized       :", q_tensor_asym.dequantize())
print()

=== torch.quantize_per_tensor (symmetric-style) ===
scale used      : 1.0259054589459276
q_tensor dtype   : torch.qint8
q_tensor.int_repr(): tensor([127, -45,   0, -23,  11,  75,  47, 126,  40,  75,  19,  30, -44, -16,
          9,  52,  87, 107, -17,   6], dtype=torch.int8)
dequantized      : tensor([130.2900, -46.1657,   0.0000, -23.5958,  11.2850,  76.9429,  48.2176,
        129.2641,  41.0362,  76.9429,  19.4922,  30.7772, -45.1398, -16.4145,
          9.2331,  53.3471,  89.2538, 109.7719, -17.4404,   6.1554])

=== torch.quantize_per_tensor (asymmetric, quint8) ===
scale, zero_point: 0.6934117335899204 67
int_repr()        : tensor([255,   0,  67,  33,  84, 178, 136, 253, 126, 177,  96, 111,   1,  44,
         80, 144, 196, 226,  41,  76], dtype=torch.uint8)
dequantized       : tensor([130.3614, -46.4586,   0.0000, -23.5760,  11.7880,  76.9687,  47.8454,
        128.9746,  40.9113,  76.2753,  20.1089,  30.5101, -45.7652, -15.9485,
          9.0144,  53.3927,  89.4501, 110.2525, -18

In [ ]:
# ---------------------------------------------------------------
# 3) Real dynamic quantization of an nn.Linear model
#    (this is the actual production PyTorch API, INT8 weights,
#     runs true integer matmul under the hood on CPU)
# ---------------------------------------------------------------
class TinyMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(20, 16)
        self.fc2 = nn.Linear(16, 4)

    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(x)))

model_fp32 = TinyMLP()
model_fp32.eval()

model_int8 = torch.ao.quantization.quantize_dynamic(
    model_fp32, {nn.Linear}, dtype=torch.qint8
)

print("=== torch.ao.quantization.quantize_dynamic ===")
print(model_int8)

x = torch.randn(2, 20)
out_fp32 = model_fp32(x)
out_int8 = model_int8(x)
rel_err = (out_fp32 - out_int8).abs().mean() / out_fp32.abs().mean()
print("\nfp32 output:", out_fp32)
print("int8 output:", out_int8)
print(f"mean relative error: {rel_err.item():.4%}")

# Inspect the actual quantized weight stored inside the layer
qlinear = model_int8.fc1
print("\nfc1 quantized weight (int_repr):")
print(qlinear.weight().int_repr())
print("fc1 weight scale:", qlinear.weight().q_scale())